In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
#CONDA ENVIRONEMNT P312

import os
#add to system path
from metrics_helpers import read_image, pre_hdr_p3, align_hdr_pred_to_gt, psnr, vsi, piqe, lpips, hdr_vdp3, pu, reinhard_tonemap, cvvdp
import os
import numpy as np
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
import pycvvdp
from metrics_helpers import initialize_fid, initialize_fvd, initialize_cvvdp
from metrics_helpers import compute_fid, fid_update, compute_fvd, fvd_update





In [17]:

pred_dir = "/data2/saikiran.tedla/hdrvideo/diff/evaluations/ours_stuttgart/over/"
gt_dir =  "/data2/saikiran.tedla/hdrvideo/diff/evaluations/stuttgart/hdr/"
video_paths = sorted(os.listdir(pred_dir))

cvvdp_metric = initialize_cvvdp()

reinhard_fvd_metric = initialize_fvd()
reinhard_fid_metric = initialize_fid()

pu_fvd_metric = initialize_fvd()
pu_fid_metric = initialize_fid()

psnr_scores = []
vsi_scores = []
piqe_scores = []
lpips_scores = []

for video_path in video_paths:
    pred_video_dir = os.path.join(pred_dir, video_path)
    gt_video_dir = os.path.join(gt_dir, video_path)
    im_paths = sorted(os.listdir(pred_video_dir))[:4]

    reinhard_preds = []
    reinhard_gts = []
    pu_preds = []
    pu_gts = []
    hdr_preds = []
    hdr_gts = []
    for im_path in im_paths:
        pred_im_path = os.path.join(pred_video_dir, im_path)
        gt_im_path = os.path.join(gt_video_dir, im_path)
        cv2_hdr_pred = read_image(pred_im_path)
        cv2_hdr_gt = read_image(gt_im_path)
        cv2_hdr_gt = pre_hdr_p3(cv2_hdr_gt)
        cv2_hdr_pred,cv2_hdr_gt,_ = align_hdr_pred_to_gt(cv2_hdr_pred, cv2_hdr_gt)
        hdr_preds.append(cv2_hdr_pred)
        hdr_gts.append(cv2_hdr_gt)

        jod = hdr_vdp3(cv2_hdr_pred, cv2_hdr_gt)

        pu_pred, pu_gt = pu(cv2_hdr_pred), pu(cv2_hdr_gt)
        pu_preds.append(pu_pred/np.max(pu_gt))
        pu_gts.append(pu_gt/np.max(pu_gt))

        reinhard_pred, reinhard_gt = reinhard_tonemap(cv2_hdr_pred), reinhard_tonemap(cv2_hdr_gt)
        reinhard_preds.append(reinhard_pred)
        reinhard_gts.append(reinhard_gt)


        pu_psnr = psnr(pu_pred, pu_gt)
        psnr_scores.append(pu_psnr)
        print("PU-PSNR:", pu_psnr)

        pu_vsi = vsi(pu_pred, pu_gt)
        vsi_scores.append(pu_vsi)   
        print("PU-VSI:", pu_vsi)

        pu_piqe = piqe(pu_pred)
        piqe_scores.append(pu_piqe)
        print("PIQE Score:", pu_piqe)

        pu_lpips = lpips(reinhard_pred, reinhard_gt)
        lpips_scores.append(pu_lpips)
        print("LPIPS Score:", pu_lpips)


        # fid_update(reinhard_pred, reinhard_gt, reinhard_fid_metric)
        # fid_update(pu_pred, pu_gt, pu_fid_metric)

    print("CVDDP Score:", cvvdp(hdr_preds, hdr_gts, cvvdp_metric))
    fvd_update(reinhard_preds, reinhard_gts, reinhard_fvd_metric)
    fvd_update(pu_preds, pu_gts, pu_fvd_metric)

print("R-FID Score:", compute_fid(reinhard_fid_metric))
print("R-FVD Score:", compute_fvd(reinhard_fvd_metric))
print("PU-FID Score:", compute_fid(pu_fid_metric))
print("PU-FVD Score:", compute_fvd(pu_fvd_metric))

print("Average PU-PSNR:", np.mean(np.array(psnr_scores)))
print("Average PU-VSI:", np.mean(np.array(vsi_scores)))
print("Average PIQE:", np.mean(np.array(piqe_scores)))
print("Average LPIPS:", np.mean(np.array(lpips_scores)))

    #add fid and fvd (then loop over videos, computer, save to csv)




Loading videomae model ...
Loading videomae model ...
PU-PSNR: 23.964846750534832
PU-VSI: 0.9589203238974816
PIQE Score: 33.20183378965893
LPIPS Score: 0.07036624103784561
PU-PSNR: 23.93992675655008
PU-VSI: 0.9591121361738444
PIQE Score: 29.06281548675388
LPIPS Score: 0.06724496930837631
PU-PSNR: 23.92404472982768
PU-VSI: 0.9591174312111899
PIQE Score: 30.461769655103097
LPIPS Score: 0.06627797335386276
PU-PSNR: 23.927466472523715
PU-VSI: 0.9593251890057951
PIQE Score: 30.28013359605343
LPIPS Score: 0.06646384298801422


100%|██████████| 1/1 [00:00<00:00,  7.62it/s]

CVDDP Score: (tensor(10., device='cuda:0'), {'Q_per_ch': array([[[[6.44708052e-07, 3.98638379e-03, 1.03511373e-02,
          4.91806585e-03, 6.01031352e-04, 7.09132291e-06,
          3.18977982e-08, 9.31322575e-10, 3.29595059e-06],
         [5.99771738e-07, 4.67919838e-03, 1.47797856e-02,
          7.66043551e-03, 4.45176382e-04, 5.38001768e-06,
          2.37487257e-08, 6.98491931e-10, 2.94670463e-06],
         [5.37373126e-07, 6.22963812e-03, 2.55402494e-02,
          1.73246302e-02, 2.68477947e-04, 2.27079727e-06,
          9.08039510e-09, 2.32830644e-10, 2.05868855e-06],
         [5.06173819e-07, 9.81065631e-03, 5.33375554e-02,
          4.73402441e-02, 8.65344657e-04, 9.20146704e-07,
          1.16415322e-09, 0.00000000e+00, 9.08039510e-07]],

        [[6.96489587e-06, 1.62411947e-04, 5.03538176e-05,
          2.97455117e-05, 3.29301693e-05, 1.15626026e-05,
          1.50618143e-06, 4.95929271e-08, 4.10326757e-05],
         [6.55301847e-06, 1.56373484e-04, 5.01894392e-05,
        

PU-PSNR: 23.22281754786325
PU-VSI: 0.9444280697011893
PIQE Score: 37.77446602219926
LPIPS Score: 0.04550519585609436
PU-PSNR: 23.21991363486625
PU-VSI: 0.944309474625646
PIQE Score: 36.197805615589104
LPIPS Score: 0.04556257650256157
PU-PSNR: 23.246971043190893
PU-VSI: 0.9443708624531345
PIQE Score: 38.73821534749287
LPIPS Score: 0.04466346651315689
PU-PSNR: 23.27602815909982
PU-VSI: 0.9444243173107094
PIQE Score: 38.38538717925371
LPIPS Score: 0.044032759964466095


100%|██████████| 1/1 [00:00<00:00,  9.61it/s]

CVDDP Score: (tensor(9.7904, device='cuda:0'), {'Q_per_ch': array([[[[4.9032364e-04, 4.9001560e-01, 2.2257452e+00, 4.8905497e+00,
          4.1969447e+00, 1.7583295e+00, 3.2104969e-01, 2.4518549e-02,
          2.2186464e-02],
         [4.6304381e-04, 4.8415151e-01, 2.2151814e+00, 4.8730426e+00,
          4.1845465e+00, 1.7421542e+00, 3.1875494e-01, 2.4341475e-02,
          2.2129830e-02],
         [4.0959031e-04, 4.7069335e-01, 2.1926982e+00, 4.8365788e+00,
          4.1594653e+00, 1.7130800e+00, 3.1373689e-01, 2.3951670e-02,
          2.1998726e-02],
         [3.4304452e-04, 4.5507175e-01, 2.1957660e+00, 4.8565593e+00,
          4.2010736e+00, 1.6932020e+00, 3.1096381e-01, 2.3831520e-02,
          2.2038618e-02]],

        [[3.3352058e-05, 1.5782164e-02, 3.6586061e-02, 9.2452623e-02,
          1.6671923e-01, 2.3217927e-01, 1.5638956e-01, 3.3959780e-02,
          2.6801316e-02],
         [3.2847049e-05, 1.5728103e-02, 3.6358360e-02, 9.2248559e-02,
          1.6653784e-01, 2.3264149e-01

PU-PSNR: 22.077808510563198
PU-VSI: 0.9793379013318259
PIQE Score: 14.086653750725203
LPIPS Score: 0.20323023200035095
PU-PSNR: 21.858654276821227
PU-VSI: 0.979954111504116
PIQE Score: 12.103156958114097
LPIPS Score: 0.2106412947177887
PU-PSNR: 21.493630940521697
PU-VSI: 0.9808564632738328
PIQE Score: 13.506861847241867
LPIPS Score: 0.2222200483083725


KeyboardInterrupt: 